# Day 49 · Multi-head, feed-forward, residual: a Transformer block

**First-principles question:** What is inside one block of a Transformer?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import urllib.request
import torch

def download(url, name):
    path = ROOT / "data" / "downloads" / name
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, path)
    return path.read_text(encoding="utf-8")

text = download("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
                "tinyshakespeare.txt")
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(1337)
print(f"{len(text):,} characters · vocabulary {vocab_size} · device {device}")

def get_batch(split, batch_size, block_size):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)
import torch.nn as nn
from torch.nn import functional as F

BATCH, BLOCK, N_EMBD, N_HEAD, N_LAYER = 32, 64, 64, 4, 2
STEPS, LR = 1000, 1e-3

## 1 · The parts
`Head` is given (yesterday's work). You write the rest.

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(N_EMBD, head_size, bias=False)
        self.query = nn.Linear(N_EMBD, head_size, bias=False)
        self.value = nn.Linear(N_EMBD, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(BLOCK, BLOCK)))

    def forward(self, x):
        B, T, C = x.shape
        k, q = self.key(x), self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = F.softmax(wei.masked_fill(self.tril[:T, :T] == 0, float("-inf")), dim=-1)
        return wei @ self.value(x)


class MultiHead(nn.Module):
    def __init__(self, n_head, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_head)])
        self.proj = nn.Linear(N_EMBD, N_EMBD)

    def forward(self, x):
        # TODO: run every head on x, join the outputs on the last dimension (torch.cat(..., dim=-1)), then self.proj
        ...

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(N_EMBD, 4 * N_EMBD), nn.ReLU(), nn.Linear(4 * N_EMBD, N_EMBD))

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.attention = MultiHead(N_HEAD, N_EMBD // N_HEAD)
        self.feed_forward = FeedForward()
        self.ln1, self.ln2 = nn.LayerNorm(N_EMBD), nn.LayerNorm(N_EMBD)

    def forward(self, x):
        # TODO: residual connections with layer norm first:
        #       x = x + self.attention(self.ln1(x)); x = x + self.feed_forward(self.ln2(x)); return x
        ...

class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, N_EMBD)
        self.position_embedding = nn.Embedding(BLOCK, N_EMBD)
        self.blocks = nn.Sequential(*[Block() for _ in range(N_LAYER)])
        self.ln_f = nn.LayerNorm(N_EMBD)
        self.lm_head = nn.Linear(N_EMBD, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding(idx) + self.position_embedding(torch.arange(T, device=idx.device))
        logits = self.lm_head(self.ln_f(self.blocks(x)))
        loss = None if targets is None else F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n_new):
        for _ in range(n_new):
            logits, _ = self(idx[:, -BLOCK:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

model = TinyGPT().to(device)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

## 2 · Before training: noise

In [ ]:
start = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(start, 200)[0].tolist()))

## 3 · Train

In [ ]:
@torch.no_grad()
def estimate_loss(iters=50):
    model.eval(); out = {}
    for split in ["train", "val"]:
        out[split] = torch.stack([model(*get_batch(split, BATCH, BLOCK))[1] for _ in range(iters)]).mean().item()
    model.train(); return out

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
history = []
for step in range(STEPS + 1):
    if step % 200 == 0:
        history.append((step, estimate_loss())); print(step, history[-1][1])
    xb, yb = get_batch("train", BATCH, BLOCK)
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()

BIGRAM_BASELINE = 2.49     # the loss of the bigram model on this text (Karpathy's video)
print("final validation loss:", history[-1][1]["val"], "· bigram baseline:", BIGRAM_BASELINE)

## 4 · After training

In [ ]:
print(decode(model.generate(start, 300)[0].tolist()))

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
xb, _ = get_batch("train", 2, BLOCK)
assert model.blocks[0].attention(model.token_embedding(xb)).shape == (2, BLOCK, N_EMBD), "MultiHead output shape"
assert history[-1][1]["val"] < BIGRAM_BASELINE, "The Transformer should beat the bigram baseline"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")